In [1]:
import sys
print(sys.executable)

C:\Users\Stoycho.rusinov\AppData\Local\anaconda3\python.exe


In [2]:
import sys

# install the package into this kernel's Python
!"{sys.executable}" -m pip install --upgrade pip
!"{sys.executable}" -m pip install playwright

# install the browser binaries Playwright needs
!"{sys.executable}" -m playwright install

In [3]:
import sys, asyncio

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [4]:
import sys, asyncio, threading

def run_coro_in_thread(coro):
    err = {}
    def runner():
        try:
            if sys.platform.startswith("win"):
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            asyncio.run(coro)
        except Exception as e:
            err["e"] = e

    t = threading.Thread(target=runner, daemon=True)
    t.start()
    t.join()

    if "e" in err:
        raise err["e"]

In [5]:
import pandas as pd
from pathlib import Path

candidates = [
    Path("nacid_nbu_results.xlsx")
]

path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("Couldn't find nacid_nbu_results.{xlsx,xlsm,xls} in the current folder.")

df = pd.read_excel(path, sheet_name="main_database")
df.columns = df.columns.astype(str).str.strip()

df["NACID"] = df["NACID"].astype(str).str.strip()
df_links = df[["NACID"]].copy()
df_links = df_links[df_links["NACID"].str.contains("ras.nacid.bg", na=False)].copy()

print("Loaded:", path.name, "| rows:", len(df), "| NACID links:", len(df_links))
df_links.head(10)

Loaded: nacid_nbu_results.xlsx | rows: 373 | NACID links: 373


,NACID
0,https://ras.nacid.bg/dissertation-preview/27099
1,https://ras.nacid.bg/dissertation-preview/16973
2,https://ras.nacid.bg/dissertation-preview/20784
3,https://ras.nacid.bg/dissertation-preview/21971
4,https://ras.nacid.bg/dissertation-preview/18966
5,https://ras.nacid.bg/dissertation-preview/21204
6,https://ras.nacid.bg/dissertation-preview/23982
7,https://ras.nacid.bg/dissertation-preview/24766
8,https://ras.nacid.bg/dissertation-preview/36650
9,https://ras.nacid.bg/dissertation-preview/15375


In [6]:
import re
import asyncio
import pandas as pd
from datetime import datetime
from pathlib import Path
from playwright.async_api import async_playwright

# =====================
# SETTINGS (EDIT THESE)
# =====================
FILE = "nacid_nbu_results.xlsx"
SHEET = "main_database"

NAME_COL = "Name"      # <-- CHANGE to your actual name column (e.g. "NAME", "Име", etc.)
LINK_COL = "NACID"

N_PEOPLE = 373
OUTFILE = "FPhys_phd_plus_headers.xlsx"

KEYWORDS = ("Асистент", "Главен асистент", "Доцент", "Професор", "Преподавател")

# =====================
# LOAD EXCEL
# =====================
path = Path(FILE)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")

df = pd.read_excel(path, sheet_name=SHEET)
df.columns = df.columns.astype(str).str.strip()

if LINK_COL not in df.columns:
    raise KeyError(f"Missing column '{LINK_COL}'. Columns present: {list(df.columns)}")

if NAME_COL not in df.columns:
    raise KeyError(
        f"Missing name column '{NAME_COL}'. "
        f"Set NAME_COL to one of: {list(df.columns)}"
    )

df[LINK_COL] = df[LINK_COL].astype(str).str.strip()
df[NAME_COL] = df[NAME_COL].astype(str).str.strip()

df = df[df[LINK_COL].str.contains("ras.nacid.bg", na=False)].copy()

# =====================
# HELPERS
# =====================
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def clean_header_prefix(s: str) -> str:
    s = norm(s)
    return re.sub(r'^[^A-Za-zА-Яа-я0-9"„]+', "", s).strip()

def first_date(text: str):
    m = re.search(r"\b(\d{1,2}\.\d{1,2}\.\d{4})\b", text or "")
    return m.group(1) if m else None

def grab_stop(label_no_colon, text):
    labels = [
        "Висше училище", "Факултет", "Първично звено", "Научна степен",
        "Професионално направление", "Диплома No/дата", "Тема на дисертационния труд"
    ]
    nxt = "|".join(map(re.escape, labels))
    m = re.search(
        rf"{re.escape(label_no_colon)}\s*:\s*(.*?)(?=\s*(?:{nxt})\s*:|$)",
        text or "",
        flags=re.S
    )
    return norm(m.group(1)) if m else None

def award_date_from_diploma(text: str):
    line = grab_stop("Диплома No/дата", text)
    return first_date(line)

def sort_date(text):
    pattern = r'\d{1,2}.\d{1,2}.\d{4}'
    match = re.search(pattern, text)
    return datetime.strptime(match.group(), "%d.%m.%Y")

# =====================
# POSITIONS HEADER EXTRACTOR (NON-CLICK)
# =====================
async def get_section_container(page, section_title: str, exact):
    title_el = page.get_by_text(section_title, exact=exact).first
    await title_el.wait_for(timeout=60000)

    section = title_el.locator("xpath=ancestor::div[contains(@class,'card')][1]")
    if await section.count() == 0:
        section = title_el.locator("xpath=ancestor::div[2]")
    return section.first

async def find_position_headers(section):
    locators = [
        section.locator("css=mat-expansion-panel-header"),
        section.locator("css=.mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel .mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel [role='button']"),
        section.locator("css=[role='button']"),
    ]
    for loc in locators:
        try:
            if await loc.count() > 0:
                return loc
        except Exception:
            continue
    return section.locator("xpath=.//*")

def canonical_header(s: str) -> str:
    s = clean_header_prefix(s)
    s = norm(s)

    # If the string accidentally contains multiple headers glued together,
    # keep only the first plausible one: "<Rank> - <Institution...>"
    # (stop at the next rank keyword if it appears again)
    for k in KEYWORDS:
        # if we see a second keyword later in the string, cut before it
        m = re.search(rf"\s({re.escape(k)}\s*-)", s)
        if m and m.start() > 0:
            s = s[:m.start()].strip()

    return s

async def get_headers_in_section(page, section_title: str):
    section = await get_section_container(page, section_title, False)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    # Prefer the true header element; fall back to class
    headers_loc = section.locator("css=mat-expansion-panel-header, .mat-expansion-panel-header")
    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()

    out = []
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue

        # keep only header-like lines
        if " - " not in raw:
            continue

        cleaned = canonical_header(raw)
        if not cleaned:
            continue
        if not any(cleaned.startswith(k) for k in KEYWORDS):
            continue

        key = cleaned.lower()
        if key in seen_keys:
            continue
        seen_keys.add(key)
        out.append(cleaned)

    return out

async def get_nested_date_in_header(page, header_title : str):
    section = await get_section_container(page, header_title, True)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    panel_heading_div = section.locator(".panel-heading")
    spanClickable = panel_heading_div.locator('.fa')
    classes = await spanClickable.get_attribute("class")
    expanded = 'fa-chevron-down' in classes

    if expanded != True:
        await spanClickable.click()

    next_div = panel_heading_div.locator("xpath=following-sibling::div[1]")
    headers_loc = next_div.locator(".row")  

    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue

        cleaned = canonical_header(raw)

        if not cleaned:
            continue

        key = cleaned.lower()

        if key in seen_keys:
            continue

        seen_keys.add(key)
        pattern = r'\d{1,2}.\d{1,2}.\d{4}'

        if "Номер/дата на акт за назначаване" in cleaned:
            date = re.search(pattern, cleaned)
            return date.group() if date else None

async def safe_goto(page, url, tries = 5):
    for attempt in range(tries):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout = 60000)
            return True
        except Exception as e:
            msg = str(e)
            if ("ERR_" in msg or "Timeout" in msg):
                await asyncio.sleep(2 * (attempt + 1))
                continue
            raise

    return False
# =====================
# PERSON SCRAPER (PHD + HEADERS)
# =====================
async def scrape_person(page, url: str, row):

    try:
        ok = await safe_goto(page, url, 5)

        if not ok:
            row["ERROR"] = "Error: Timeout waiting for page to be loaded"
            return '', '', '', '', ''
        
    except Exception as e:
        row["ERROR"] = str(e)
        return '', '', '', '', ''
    
    phd = {
        'Phd_Institution': '',
        'PhD_Faculty': '',
        'PhD_PrimaryUnit': '',
        'PhD_Degree': '',
        'PhD_Field': '',
        'PhD_DiplomaNoDate': '',
        'PhD_AwardDate': '',
        'Dissertation_Title': ''
    }

    current_headers = []
    past_headers = []
    current_nested_headers = []
    past_nested_headers = []
         
    try:
        # ----- PhD block -----
        loc = page.locator("h4.panel-title", has_text = "Научни степени")
        await loc.wait_for(state = "visible")
        next_div = loc.locator("xpath=../following-sibling::div[1]")
        divPanels = await next_div.locator("div.panel-default").all()

        for div_panel in divPanels:
            clickable = div_panel.locator('.btn-block')
            await clickable.click()
            await div_panel.get_by_text("Диплома No/дата:", exact=False).wait_for(timeout=60000)

            anchor = div_panel.get_by_text("Диплома No/дата:", exact=False).first
            best_text, best_score = "", -1
            target_labels = [
                "Висше училище:", "Факултет:", "Първично звено:", "Научна степен:",
                "Професионално направление:", "Диплома No/дата:", "Тема на дисертационния труд:"
            ]
            for depth in range(1, 15):
                loc = anchor.locator(f"xpath=ancestor::div[{depth}]")
                if await loc.count() == 0:
                    break
                txt = await loc.first.inner_text()
                score = sum(1 for lab in target_labels if lab in txt)
                if score > best_score:
                    best_score, best_text = score, txt

            phd_block = best_text.replace("\u00a0", " ").strip()

            phd["Phd_Institution"] = phd["Phd_Institution"] + (grab_stop("Висше училище", phd_block) or '') + ' '
            phd["PhD_Faculty"] = phd["PhD_Faculty"] +  (grab_stop("Факултет", phd_block) or '') + ' '
            phd["PhD_PrimaryUnit"] = phd["PhD_PrimaryUnit"] +  (grab_stop("Първично звено", phd_block) or '') + ' '
            phd["PhD_Degree"] = phd["PhD_Degree"] +  (grab_stop("Научна степен", phd_block) or '') + ' '
            phd["PhD_Field"] = phd["PhD_Field"] +  (grab_stop("Професионално направление", phd_block) or '') + ' '
            phd["PhD_DiplomaNoDate"] = phd["PhD_DiplomaNoDate"] +  (grab_stop("Диплома No/дата", phd_block) or '') + ' '
            phd["PhD_AwardDate"] = phd["PhD_AwardDate"] +  (award_date_from_diploma(phd_block) or '') + ' '
            phd["Dissertation_Title"] = phd["Dissertation_Title"] +  (grab_stop("Тема на дисертационния труд", phd_block) or '') + ' '

    except Exception as e:
        row["ERROR"] = row.get("ERROR", " ") + str(e)

    try:
        # ----- Positions headers -----
        current_headers = await get_headers_in_section(page, "Настоящи академични длъжности")
        past_headers = await get_headers_in_section(page, "Заемани академични длъжности")

        current_nested_headers = []
        past_nested_headers = []

        for current_header in current_headers:
            date = await get_nested_date_in_header(page, current_header)
            result = current_header + ', ' + date
            current_nested_headers.append(result)

        for current_past_header in past_headers:
            past_date = await get_nested_date_in_header(page, current_past_header)
            result = current_past_header + ', ' + past_date
            past_nested_headers.append(result)
        
        current_nested_headers = sorted(current_nested_headers, key = sort_date)
        past_nested_headers = sorted(past_nested_headers, key = sort_date)
    except Exception as e:
       row["ERROR"] = row.get("ERROR", " ") + str(e)


    return phd, " | ".join(current_headers), " | ".join(past_headers), " | ".join(current_nested_headers), " | ".join(past_nested_headers)

# =====================
# RUN MANY PEOPLE + SIMPLE PROGRESS
# =====================
# async def run(df):
async def run(df):
    subset = df[[NAME_COL, LINK_COL]].dropna().head(N_PEOPLE).copy()
    rows = []
    total = len(subset)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, (name, url) in enumerate(zip(subset[NAME_COL], subset[LINK_COL]), start=1):
            print(f"[{i}/{total}] {url}")

            row = {"Name": name, "NACID_URL": url}
            phd, cur_headers, past_headers, cur_nested_headers, past_nested_headers = await scrape_person(page, url, row)
            row.update(phd)
            row["Current_Positions_Headers"] = cur_headers
            row["Past_Positions_Headers"] = past_headers
            row['Current_Positions'] = cur_nested_headers
            row["Past_Positions"] = past_nested_headers
            rows.append(row)
            await asyncio.sleep(0.6)

        await browser.close()

    out = pd.DataFrame(rows)
    out.to_excel(OUTFILE, index=False)
    print(f"\nDONE → {OUTFILE}")

# =====================
# NOTEBOOK RUN
# =====================
run_coro_in_thread(run(df))



[1/373] https://ras.nacid.bg/dissertation-preview/27099
[2/373] https://ras.nacid.bg/dissertation-preview/16973
[3/373] https://ras.nacid.bg/dissertation-preview/20784
[4/373] https://ras.nacid.bg/dissertation-preview/21971
[5/373] https://ras.nacid.bg/dissertation-preview/18966
[6/373] https://ras.nacid.bg/dissertation-preview/21204
[7/373] https://ras.nacid.bg/dissertation-preview/23982
[8/373] https://ras.nacid.bg/dissertation-preview/24766
[9/373] https://ras.nacid.bg/dissertation-preview/36650
[10/373] https://ras.nacid.bg/dissertation-preview/15375
[11/373] https://ras.nacid.bg/dissertation-preview/75846
[12/373] https://ras.nacid.bg/dissertation-preview/18993
[13/373] https://ras.nacid.bg/dissertation-preview/31827
[14/373] https://ras.nacid.bg/dissertation-preview/16810
[15/373] https://ras.nacid.bg/dissertation-preview/20600
[16/373] https://ras.nacid.bg/dissertation-preview/15364
[17/373] https://ras.nacid.bg/dissertation-preview/17587
[18/373] https://ras.nacid.bg/dissertati

In [ ]:
import re
import asyncio
import pandas as pd
from datetime import datetime
from pathlib import Path
from playwright.async_api import async_playwright

# =====================
# SETTINGS (EDIT THESE)
# =====================
FILE = "inbreeding_at_FPhys.xlsx"
SHEET = "main_database"

NAME_COL = "Name"      # <-- CHANGE to your actual name column (e.g. "NAME", "Име", etc.)
LINK_COL = "NACID"

N_PEOPLE = 112
OUTFILE = "FPhys_phd_plus_headers.xlsx"

KEYWORDS = ("Асистент", "Главен асистент", "Доцент", "Професор", "Преподавател")

# =====================
# LOAD EXCEL
# =====================
path = Path(FILE)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")

df = pd.read_excel(path, sheet_name=SHEET)
df.columns = df.columns.astype(str).str.strip()

if LINK_COL not in df.columns:
    raise KeyError(f"Missing column '{LINK_COL}'. Columns present: {list(df.columns)}")

if NAME_COL not in df.columns:
    raise KeyError(
        f"Missing name column '{NAME_COL}'. "
        f"Set NAME_COL to one of: {list(df.columns)}"
    )

df[LINK_COL] = df[LINK_COL].astype(str).str.strip()
df[NAME_COL] = df[NAME_COL].astype(str).str.strip()

df = df[df[LINK_COL].str.contains("ras.nacid.bg", na=False)].copy()

# =====================
# HELPERS
# =====================
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def clean_header_prefix(s: str) -> str:
    s = norm(s)
    return re.sub(r'^[^A-Za-zА-Яа-я0-9"„]+', "", s).strip()

def first_date(text: str):
    m = re.search(r"\b(\d{1,2}\.\d{1,2}\.\d{4})\b", text or "")
    return m.group(1) if m else None

def grab_stop(label_no_colon, text):
    labels = [
        "Висше училище", "Факултет", "Първично звено", "Научна степен",
        "Професионално направление", "Диплома No/дата", "Тема на дисертационния труд"
    ]
    nxt = "|".join(map(re.escape, labels))
    m = re.search(
        rf"{re.escape(label_no_colon)}\s*:\s*(.*?)(?=\s*(?:{nxt})\s*:|$)",
        text or "",
        flags=re.S
    )
    return norm(m.group(1)) if m else None

def award_date_from_diploma(text: str):
    line = grab_stop("Диплома No/дата", text)
    return first_date(line)

def sort_date(text):
    pattern = r'\d{1,2}.\d{1,2}.\d{4}'
    match = re.search(pattern, text)
    return datetime.strptime(match.group(), "%d.%m.%Y")

# =====================
# POSITIONS HEADER EXTRACTOR (NON-CLICK)
# =====================
async def get_section_container(page, section_title: str, exact):
    title_el = page.get_by_text(section_title, exact=exact).first
    await title_el.wait_for(timeout=60000)

    section = title_el.locator("xpath=ancestor::div[contains(@class,'card')][1]")
    if await section.count() == 0:
        section = title_el.locator("xpath=ancestor::div[2]")
    return section.first

async def find_position_headers(section):
    locators = [
        section.locator("css=mat-expansion-panel-header"),
        section.locator("css=.mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel .mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel [role='button']"),
        section.locator("css=[role='button']"),
    ]
    for loc in locators:
        try:
            if await loc.count() > 0:
                return loc
        except Exception:
            continue
    return section.locator("xpath=.//*")

def canonical_header(s: str) -> str:
    s = clean_header_prefix(s)
    s = norm(s)

    # If the string accidentally contains multiple headers glued together,
    # keep only the first plausible one: "<Rank> - <Institution...>"
    # (stop at the next rank keyword if it appears again)
    for k in KEYWORDS:
        # if we see a second keyword later in the string, cut before it
        m = re.search(rf"\s({re.escape(k)}\s*-)", s)
        if m and m.start() > 0:
            s = s[:m.start()].strip()

    return s

async def get_headers_in_section(page, section_title: str):
    section = await get_section_container(page, section_title, False)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    # Prefer the true header element; fall back to class
    headers_loc = section.locator("css=mat-expansion-panel-header, .mat-expansion-panel-header")
    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()

    out = []
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue

        # keep only header-like lines
        if " - " not in raw:
            continue

        cleaned = canonical_header(raw)
        if not cleaned:
            continue
        if not any(cleaned.startswith(k) for k in KEYWORDS):
            continue

        key = cleaned.lower()
        if key in seen_keys:
            continue
        seen_keys.add(key)
        out.append(cleaned)

    return out

async def get_nested_date_in_header(page, header_title : str):
    section = await get_section_container(page, header_title, True)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    panel_heading_div = section.locator(".panel-heading")
    spanClickable = panel_heading_div.locator('.fa')
    classes = await spanClickable.get_attribute("class")
    expanded = 'fa-chevron-down' in classes

    if expanded != True:
        await spanClickable.click()

    next_div = panel_heading_div.locator("xpath=following-sibling::div[1]")
    headers_loc = next_div.locator(".row")  

    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue

        cleaned = canonical_header(raw)

        if not cleaned:
            continue

        key = cleaned.lower()

        if key in seen_keys:
            continue

        seen_keys.add(key)
        pattern = r'\d{1,2}.\d{1,2}.\d{4}'

        if "Номер/дата на акт за назначаване" in cleaned:
            date = re.search(pattern, cleaned)
            return date.group() if date else None

async def safe_goto(page, url, tries = 5):
    for attempt in range(tries):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout = 60000)
            return True
        except Exception as e:
            msg = str(e)
            if ("ERR_" in msg or "Timeout" in msg):
                await asyncio.sleep(2 * (attempt + 1))
                continue
            raise

    return False
# =====================
# PERSON SCRAPER (PHD + HEADERS)
# =====================
async def scrape_person(page, url: str, row):

    try:
        ok = await safe_goto(page, url, 5)

        if not ok:
            row["ERROR"] = "Error: Timeout waiting for page to be loaded"
            return '', '', '', '', ''
        
    except Exception as e:
        row["ERROR"] = str(e)
        return '', '', '', '', ''
    
    phd = {
        'Phd_Institution': '',
        'PhD_Faculty': '',
        'PhD_PrimaryUnit': '',
        'PhD_Degree': '',
        'PhD_Field': '',
        'PhD_DiplomaNoDate': '',
        'PhD_AwardDate': '',
        'Dissertation_Title': ''
    }

    current_headers = []
    past_headers = []
    current_nested_headers = []
    past_nested_headers = []
         
    try:
        # ----- PhD block -----
        loc = page.locator("h4.panel-title", has_text = "Научни степени")
        await loc.wait_for(state = "visible")
        next_div = loc.locator("xpath=../following-sibling::div[1]")
        divPanels = await next_div.locator("div.panel-default").all()

        for div_panel in divPanels:
            clickable = div_panel.locator('.btn-block')
            await clickable.click()
            await div_panel.get_by_text("Диплома No/дата:", exact=False).wait_for(timeout=60000)

            anchor = div_panel.get_by_text("Диплома No/дата:", exact=False).first
            best_text, best_score = "", -1
            target_labels = [
                "Висше училище:", "Факултет:", "Първично звено:", "Научна степен:",
                "Професионално направление:", "Диплома No/дата:", "Тема на дисертационния труд:"
            ]
            for depth in range(1, 15):
                loc = anchor.locator(f"xpath=ancestor::div[{depth}]")
                if await loc.count() == 0:
                    break
                txt = await loc.first.inner_text()
                score = sum(1 for lab in target_labels if lab in txt)
                if score > best_score:
                    best_score, best_text = score, txt

            phd_block = best_text.replace("\u00a0", " ").strip()

            phd["Phd_Institution"] = phd["Phd_Institution"] + (grab_stop("Висше училище", phd_block) or '') + ' '
            phd["PhD_Faculty"] = phd["PhD_Faculty"] +  (grab_stop("Факултет", phd_block) or '') + ' '
            phd["PhD_PrimaryUnit"] = phd["PhD_PrimaryUnit"] +  (grab_stop("Първично звено", phd_block) or '') + ' '
            phd["PhD_Degree"] = phd["PhD_Degree"] +  (grab_stop("Научна степен", phd_block) or '') + ' '
            phd["PhD_Field"] = phd["PhD_Field"] +  (grab_stop("Професионално направление", phd_block) or '') + ' '
            phd["PhD_DiplomaNoDate"] = phd["PhD_DiplomaNoDate"] +  (grab_stop("Диплома No/дата", phd_block) or '') + ' '
            phd["PhD_AwardDate"] = phd["PhD_AwardDate"] +  (award_date_from_diploma(phd_block) or '') + ' '
            phd["Dissertation_Title"] = phd["Dissertation_Title"] +  (grab_stop("Тема на дисертационния труд", phd_block) or '') + ' '

    except Exception as e:
        row["ERROR"] = row.get("ERROR", " ") + str(e)

    try:
        # ----- Positions headers -----
        current_headers = await get_headers_in_section(page, "Настоящи академични длъжности")
        past_headers = await get_headers_in_section(page, "Заемани академични длъжности")

        current_nested_headers = []
        past_nested_headers = []

        for current_header in current_headers:
            date = await get_nested_date_in_header(page, current_header)
            result = current_header + ', ' + date
            current_nested_headers.append(result)

        for current_past_header in past_headers:
            past_date = await get_nested_date_in_header(page, current_past_header)
            result = current_past_header + ', ' + past_date
            past_nested_headers.append(result)
        
        current_nested_headers = sorted(current_nested_headers, key = sort_date)
        past_nested_headers = sorted(past_nested_headers, key = sort_date)
    except Exception as e:
       row["ERROR"] = row.get("ERROR", " ") + str(e)


    return phd, " | ".join(current_headers), " | ".join(past_headers), " | ".join(current_nested_headers), " | ".join(past_nested_headers)

# =====================
# RUN MANY PEOPLE + SIMPLE PROGRESS
# =====================
# async def run(df):
async def run(df):
    subset = df[[NAME_COL, LINK_COL]].dropna().head(N_PEOPLE).copy()
    rows = []
    total = len(subset)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, (name, url) in enumerate(zip(subset[NAME_COL], subset[LINK_COL]), start=1):
            print(f"[{i}/{total}] {url}")

            row = {"Name": name, "NACID_URL": url}
            phd, cur_headers, past_headers, cur_nested_headers, past_nested_headers = await scrape_person(page, url, row)
            row.update(phd)
            row["Current_Positions_Headers"] = cur_headers
            row["Past_Positions_Headers"] = past_headers
            row['Current_Positions'] = cur_nested_headers
            row["Past_Positions"] = past_nested_headers
            rows.append(row)
            await asyncio.sleep(0.6)

        await browser.close()

    out = pd.DataFrame(rows)
    out.to_excel(OUTFILE, index=False)
    print(f"\nDONE → {OUTFILE}")

# =====================
# NOTEBOOK RUN
# =====================
run_coro_in_thread(run(df))

Check specifically whether the person studied outside of Bulgaria

In [8]:
import re
import asyncio
import pandas as pd
from datetime import datetime
from pathlib import Path
from playwright.async_api import async_playwright

# =====================
# SETTINGS (EDIT THESE)
# =====================
FILE = "nacid_nbu_results.xlsx"
SHEET = "main_database"

NAME_COL = "Name"
LINK_COL = "NACID"

N_PEOPLE = 373

KEYWORDS = ("Асистент", "Главен асистент", "Доцент", "Професор", "Преподавател")

# =====================
# LOAD EXCEL
# =====================
path = Path(FILE)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")

df = pd.read_excel(path, sheet_name=SHEET)
df.columns = df.columns.astype(str).str.strip()

if LINK_COL not in df.columns:
    raise KeyError(f"Missing column '{LINK_COL}'. Columns present: {list(df.columns)}")

if NAME_COL not in df.columns:
    raise KeyError(
        f"Missing name column '{NAME_COL}'. "
        f"Set NAME_COL to one of: {list(df.columns)}"
    )

df[LINK_COL] = df[LINK_COL].astype(str).str.strip()
df[NAME_COL] = df[NAME_COL].astype(str).str.strip()

df = df[df[LINK_COL].str.contains("ras.nacid.bg", na=False)].copy()

# =====================
# HELPERS
# =====================
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def clean_header_prefix(s: str) -> str:
    s = norm(s)
    return re.sub(r'^[^A-Za-zА-Яа-я0-9"„]+', "", s).strip()

def first_date(text: str):
    m = re.search(r"\b(\d{1,2}\.\d{1,2}\.\d{4})\b", text or "")
    return m.group(1) if m else None

def grab_stop(label_no_colon, text):
    labels = [
        "Висше училище", "Факултет", "Първично звено", "Научна степен",
        "Професионално направление", "Диплома No/дата", "Тема на дисертационния труд",
        # keep this list as-is for stop conditions; we’re not adding "Защитил в чужбина" here
        # because we only want to *detect presence* anywhere in the phd block
    ]
    nxt = "|".join(map(re.escape, labels))
    m = re.search(
        rf"{re.escape(label_no_colon)}\s*:\s*(.*?)(?=\s*(?:{nxt})\s*:|$)",
        text or "",
        flags=re.S
    )
    return norm(m.group(1)) if m else None

def award_date_from_diploma(text: str):
    line = grab_stop("Диплома No/дата", text)
    return first_date(line)

def sort_date(text):
    pattern = r'\d{1,2}.\d{1,2}.\d{4}'
    match = re.search(pattern, text)
    return datetime.strptime(match.group(), "%d.%m.%Y")

# =====================
# POSITIONS HEADER EXTRACTOR (NON-CLICK)
# =====================
async def get_section_container(page, section_title: str, exact):
    title_el = page.get_by_text(section_title, exact=exact).first
    await title_el.wait_for(timeout=60000)

    section = title_el.locator("xpath=ancestor::div[contains(@class,'card')][1]")
    if await section.count() == 0:
        section = title_el.locator("xpath=ancestor::div[2]")
    return section.first

async def find_position_headers(section):
    locators = [
        section.locator("css=mat-expansion-panel-header"),
        section.locator("css=.mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel .mat-expansion-panel-header"),
        section.locator("css=mat-expansion-panel [role='button']"),
        section.locator("css=[role='button']"),
    ]
    for loc in locators:
        try:
            if await loc.count() > 0:
                return loc
        except Exception:
            continue
    return section.locator("xpath=.//*")

def canonical_header(s: str) -> str:
    s = clean_header_prefix(s)
    s = norm(s)

    for k in KEYWORDS:
        m = re.search(rf"\s({re.escape(k)}\s*-)", s)
        if m and m.start() > 0:
            s = s[:m.start()].strip()

    return s

async def get_headers_in_section(page, section_title: str):
    section = await get_section_container(page, section_title, False)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    headers_loc = section.locator("css=mat-expansion-panel-header, .mat-expansion-panel-header")
    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()

    out = []
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue
        if " - " not in raw:
            continue

        cleaned = canonical_header(raw)
        if not cleaned:
            continue
        if not any(cleaned.startswith(k) for k in KEYWORDS):
            continue

        key = cleaned.lower()
        if key in seen_keys:
            continue
        seen_keys.add(key)
        out.append(cleaned)

    return out

async def get_nested_date_in_header(page, header_title: str):
    section = await get_section_container(page, header_title, True)
    await section.scroll_into_view_if_needed()
    await asyncio.sleep(0.2)

    panel_heading_div = section.locator(".panel-heading")
    spanClickable = panel_heading_div.locator('.fa')
    classes = await spanClickable.get_attribute("class")
    expanded = 'fa-chevron-down' in (classes or "")

    if expanded != True:
        await spanClickable.click()

    next_div = panel_heading_div.locator("xpath=following-sibling::div[1]")
    headers_loc = next_div.locator(".row")

    if await headers_loc.count() == 0:
        headers_loc = await find_position_headers(section)

    n = await headers_loc.count()
    seen_keys = set()

    for i in range(n):
        h = headers_loc.nth(i)
        raw = norm(await h.inner_text())
        if not raw:
            continue

        cleaned = canonical_header(raw)
        if not cleaned:
            continue

        key = cleaned.lower()
        if key in seen_keys:
            continue
        seen_keys.add(key)

        pattern = r'\d{1,2}.\d{1,2}.\d{4}'
        if "Номер/дата на акт за назначаване" in cleaned:
            date = re.search(pattern, cleaned)
            return date.group() if date else None

    return None

async def safe_goto(page, url, tries=5):
    for attempt in range(tries):
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            return True
        except Exception as e:
            msg = str(e)
            if ("ERR_" in msg or "Timeout" in msg):
                await asyncio.sleep(2 * (attempt + 1))
                continue
            raise
    return False

# =====================
# PERSON SCRAPER (PHD + HEADERS) + "Защитил в чужбина" CHECK
# =====================
async def scrape_person(page, url: str, row):
    # Returns: True/False (field present in ANY PhD panel), or None on fatal error
    try:
        ok = await safe_goto(page, url, 5)
        if not ok:
            row["ERROR"] = "Timeout loading page"
            return None
    except Exception as e:
        row["ERROR"] = str(e)
        return None

    defended_abroad_field_present = False

    try:
        # ----- PhD block -----
        loc = page.locator("h4.panel-title", has_text="Научни степени")
        await loc.wait_for(state="visible", timeout=60000)

        next_div = loc.locator("xpath=../following-sibling::div[1]")
        divPanels = await next_div.locator("div.panel-default").all()

        for div_panel in divPanels:
            clickable = div_panel.locator(".btn-block").first

            # expand only if not expanded (avoids toggling closed)
            aria = await clickable.get_attribute("aria-expanded")
            if aria != "true":
                await clickable.click()

            # wait for typical content inside the expanded panel (more reliable than diploma only)
            await div_panel.get_by_text("Научна степен", exact=False).wait_for(timeout=60000)

            # IMPORTANT: scan the ENTIRE panel text (this includes "Защитил в чужбина")
            panel_text = (await div_panel.inner_text()).replace("\u00a0", " ")

            if re.search(r"\bЗащитил\s+в\s+чужбина\b", panel_text, flags=re.IGNORECASE):
                defended_abroad_field_present = True
                # no need to keep scanning other phd panels
                break

    except Exception as e:
        row["ERROR"] = (row.get("ERROR", "") + " " + str(e)).strip()

    return defended_abroad_field_present


async def run(df):
    subset = df[[NAME_COL, LINK_COL]].dropna().head(N_PEOPLE).copy()
    total = len(subset)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for i, (name, url) in enumerate(zip(subset[NAME_COL], subset[LINK_COL]), start=1):
            # progress first (as before)
            print(f"[{i}/{total}] {url}")

            row = {"Name": name, "NACID_URL": url}
            present = await scrape_person(page, url, row)

            if present is None:
                print(f"{name} - ERROR: {row.get('ERROR','(unknown)')}")
            else:
                print(f"{name} - {'YES' if present else 'NO'}")

            await asyncio.sleep(0.6)

        await browser.close()

# =====================
# NOTEBOOK RUN
# =====================
# If you already have run_coro_in_thread(...) in your notebook, keep using it.
# Otherwise, use this simple fallback:
def run_coro_in_thread(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        import threading
        def runner():
            asyncio.run(coro)
        t = threading.Thread(target=runner, daemon=True)
        t.start()
    else:
        asyncio.run(coro)

run_coro_in_thread(run(df))

[1/373] https://ras.nacid.bg/dissertation-preview/27099
Абайнех Асефа Шенкута - NO
[2/373] https://ras.nacid.bg/dissertation-preview/16973
Абас Ахмед Абдул - NO
[3/373] https://ras.nacid.bg/dissertation-preview/20784
Абас Хасан Ал Рамзи - NO
[4/373] https://ras.nacid.bg/dissertation-preview/21971
Аббас Али Мохамед Рамадан - NO
[5/373] https://ras.nacid.bg/dissertation-preview/18966
Аббас Мансур Ал Хасан - NO
[6/373] https://ras.nacid.bg/dissertation-preview/21204
Аббас Хамуди Батах - NO
[7/373] https://ras.nacid.bg/dissertation-preview/23982
Аббас Хафет Аббас - NO
[8/373] https://ras.nacid.bg/dissertation-preview/24766
Аббас Неджар - NO
[9/373] https://ras.nacid.bg/dissertation-preview/36650
Абд Ал-Насър Мохамед Насър - NO
[10/373] https://ras.nacid.bg/dissertation-preview/15375
Абдала Растанауи - NO
[11/373] https://ras.nacid.bg/dissertation-preview/75846
Абдала Хаджайрех - NO
[12/373] https://ras.nacid.bg/dissertation-preview/18993
Абдел Максуд Абдела Иса - NO
[13/373] https://ras.na